# Nashville Housing Data Cleaning

Initial data loading, inspection, and setup for the Nashville Housing dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load the data

In [2]:
# read in the raw data and create a copy for cleaning
RAW_PATH = Path("../data/raw/Nashville Housing Data for Data Cleaning.xlsx")
df_raw = pd.read_excel(RAW_PATH)
df_clean = df_raw.copy()

# create a dataframe to log changes
change_log = pd.DataFrame(columns=['step', 'description', 'rows_affected'])

# add a function to log changes to the change_log dataframe
def log_change(step: str, description: str, rows_affected: int) -> None:
    """Append one row to change_log (step, description, rows_affected)."""
    change_log.loc[len(change_log)] = [step, description, int(rows_affected)]

In [3]:
print("--- df.head() ---")
display(df_clean.head())

--- df.head() ---


,UniqueID,ParcelID,LandUse,PropertyAddress,SaleDate,SalePrice,LegalReference,SoldAsVacant,OwnerName,OwnerAddress,Acreage,TaxDistrict,LandValue,BuildingValue,TotalValue,YearBuilt,Bedrooms,FullBath,HalfBath
0,2045,007 00 0 125.00,SINGLE FAMILY,"1808 FOX CHASE DR, GOODLETTSVILLE",2013-04-09,240000,20130412-0036474,No,"FRAZIER, CYRENTHA LYNETTE","1808 FOX CHASE DR, GOODLETTSVILLE, TN",2.3,GENERAL SERVICES DISTRICT,50000.0,168200.0,235700.0,1986.0,3.0,3.0,0.0
1,16918,007 00 0 130.00,SINGLE FAMILY,"1832 FOX CHASE DR, GOODLETTSVILLE",2014-06-10,366000,20140619-0053768,No,"BONER, CHARLES & LESLIE","1832 FOX CHASE DR, GOODLETTSVILLE, TN",3.5,GENERAL SERVICES DISTRICT,50000.0,264100.0,319000.0,1998.0,3.0,3.0,2.0
2,54582,007 00 0 138.00,SINGLE FAMILY,"1864 FOX CHASE DR, GOODLETTSVILLE",2016-09-26,435000,20160927-0101718,No,"WILSON, JAMES E. & JOANNE","1864 FOX CHASE DR, GOODLETTSVILLE, TN",2.9,GENERAL SERVICES DISTRICT,50000.0,216200.0,298000.0,1987.0,4.0,3.0,0.0
3,43070,007 00 0 143.00,SINGLE FAMILY,"1853 FOX CHASE DR, GOODLETTSVILLE",2016-01-29,255000,20160129-0008913,No,"BAKER, JAY K. & SUSAN E.","1853 FOX CHASE DR, GOODLETTSVILLE, TN",2.6,GENERAL SERVICES DISTRICT,50000.0,147300.0,197300.0,1985.0,3.0,3.0,0.0
4,22714,007 00 0 149.00,SINGLE FAMILY,"1829 FOX CHASE DR, GOODLETTSVILLE",2014-10-10,278000,20141015-0095255,No,"POST, CHRISTOPHER M. & SAMANTHA C.","1829 FOX CHASE DR, GOODLETTSVILLE, TN",2.0,GENERAL SERVICES DISTRICT,50000.0,152300.0,202300.0,1984.0,4.0,3.0,0.0


In [4]:
print("--- df_clean.info() ---")
df_clean.info()

--- df_clean.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 56477 entries, 0 to 56476
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   UniqueID         56477 non-null  int64         
 1   ParcelID         56477 non-null  str           
 2   LandUse          56477 non-null  str           
 3   PropertyAddress  56448 non-null  str           
 4   SaleDate         56477 non-null  datetime64[us]
 5   SalePrice        56477 non-null  int64         
 6   LegalReference   56477 non-null  object        
 7   SoldAsVacant     56477 non-null  str           
 8   OwnerName        25261 non-null  str           
 9   OwnerAddress     26015 non-null  str           
 10  Acreage          26015 non-null  float64       
 11  TaxDistrict      26015 non-null  str           
 12  LandValue        26015 non-null  float64       
 13  BuildingValue    26015 non-null  float64       
 14  TotalValue       26015 no

In [5]:
print("--- df.shape ---")
print(df_clean.shape)

--- df.shape ---
(56477, 19)


In [6]:
assert df_clean.shape == (56477, 19), f"Unexpected shape: {df_clean.shape}"
assert 'df_raw' in globals() and 'df_clean' in globals(), "Dataframes not initialized"
print("Initial loading and inspection assertions passed successfully.")

Initial loading and inspection assertions passed successfully.


## 2. Data Profiling

In [7]:
# Missing values: count and percentage per column
missing = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_pct": (df_clean.isna().mean() * 100).round(2),
})
missing[missing["missing_count"] > 0].sort_values("missing_count", ascending=False)

,missing_count,missing_pct
HalfBath,32333,57.25
Bedrooms,32320,57.23
YearBuilt,32314,57.22
FullBath,32202,57.02
OwnerName,31216,55.27
TotalValue,30462,53.94
OwnerAddress,30462,53.94
Acreage,30462,53.94
LandValue,30462,53.94
BuildingValue,30462,53.94


In [8]:
# Duplicates at three levels of strictness
print("Fully identical rows:", df_clean.duplicated().sum())
print("Repeated UniqueID:", df_clean["UniqueID "].duplicated().sum())

# Same sale recorded more than once (business-key duplicates)
key_cols = ["ParcelID", "PropertyAddress", "SaleDate", "SalePrice", "LegalReference"]
print("Duplicates on business key:", df_clean.duplicated(subset=key_cols).sum())

Fully identical rows: 0
Repeated UniqueID: 0
Duplicates on business key: 103


In [9]:
# Categorical consistency
print(df_clean["SoldAsVacant"].value_counts(dropna=False))
print(df_clean["LandUse"].nunique(), "distinct LandUse values")

SoldAsVacant
No     51403
Yes     4623
N        399
Y         52
Name: count, dtype: int64
39 distinct LandUse values


In [10]:
# Hidden whitespace in address columns
for col in ["PropertyAddress", "OwnerAddress"]:
    s = df_clean[col].dropna().astype(str)
    print(col,
          "| double spaces:", s.str.contains(r"\s{2,}").sum(),
          "| leading/trailing:", (s != s.str.strip()).sum())

PropertyAddress | double spaces: 55863 | leading/trailing: 7
OwnerAddress | double spaces: 25315 | leading/trailing: 0


In [11]:
# Numeric sanity check
df_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
UniqueID,56477.0,28334.001133,0.0,14186.0,28313.0,42513.0,56635.0,16352.590651
SaleDate,56477,2015-02-08 08:02:27.373267,2013-01-02 00:00:00,2014-04-11 00:00:00,2015-03-24 00:00:00,2015-12-30 00:00:00,2019-12-13 00:00:00,NaN
SalePrice,56477.0,327226.369566,50.0,135000.0,205100.0,328500.0,54278060.0,929832.611076
Acreage,26015.0,0.498923,0.01,0.18,0.27,0.45,160.06,1.570454
LandValue,26015.0,69068.557601,100.0,21000.0,28800.0,60000.0,2772000.0,106040.12844
BuildingValue,26015.0,160784.677109,0.0,75900.0,111400.0,180700.0,12971800.0,206799.854557
TotalValue,26015.0,232375.398847,100.0,102800.0,148500.0,268350.0,13940400.0,281064.346448
YearBuilt,24163.0,1963.744899,1799.0,1948.0,1960.0,1983.0,2017.0,26.542982
Bedrooms,24157.0,3.089912,0.0,3.0,3.0,3.0,11.0,0.852869
FullBath,24275.0,1.886014,0.0,1.0,2.0,2.0,10.0,0.961515


In [12]:
# Check 1: are the six "30,462 missing" columns missing in the SAME rows?
struct_cols = ["OwnerAddress", "Acreage", "LandValue",
               "BuildingValue", "TotalValue", "TaxDistrict"]
pattern = df_clean[struct_cols].isna()
print("Rows where ALL six are missing:", pattern.all(axis=1).sum())
print("Rows where SOME (not all) are missing:",
      (pattern.any(axis=1) & ~pattern.all(axis=1)).sum())

# Check 2: do the 103 duplicates differ ONLY by UniqueID?
key_cols = ["ParcelID", "PropertyAddress", "SaleDate", "SalePrice", "LegalReference"]
dups = df_clean[df_clean.duplicated(subset=key_cols, keep=False)].sort_values(key_cols)
other_cols = [c for c in df_clean.columns if c not in ["UniqueID "]]
print("Duplicate rows identical except UniqueID:",
      df_clean.duplicated(subset=other_cols, keep=False).sum(), "of", len(dups))

# Check 3: how many of the 29 missing addresses can be recovered via ParcelID?
known = (df_clean.dropna(subset=["PropertyAddress"])
           .drop_duplicates("ParcelID")
           .set_index("ParcelID")["PropertyAddress"])
missing_addr = df_clean[df_clean["PropertyAddress"].isna()]
print("Recoverable via ParcelID:",
      missing_addr["ParcelID"].isin(known.index).sum(), "of", len(missing_addr))

Rows where ALL six are missing: 30462
Rows where SOME (not all) are missing: 0
Duplicate rows identical except UniqueID: 204 of 206
Recoverable via ParcelID: 29 of 29


In [13]:
# Find duplicate groups (by business key) that are NOT identical in other columns
key_cols = ["ParcelID", "PropertyAddress", "SaleDate", "SalePrice", "LegalReference"]
dups = df_clean[df_clean.duplicated(subset=key_cols, keep=False)].sort_values(key_cols)

# Rows identical except UniqueID are "safe" duplicates; the rest need a manual look
other_cols = [c for c in df_clean.columns if c != "UniqueID "]
safe_mask = dups.duplicated(subset=other_cols, keep=False)
odd_rows = dups[~safe_mask]

# Show the odd rows transposed so differences are easy to read
odd_rows.T

,30034,30035
UniqueID,12629,11912
ParcelID,110 03 0A 005.00,110 03 0A 005.00
LandUse,SINGLE FAMILY,VACANT RES LAND
PropertyAddress,"2616 LAKESIDE MEADOWS DR, MOUNT JULIET","2616 LAKESIDE MEADOWS DR, MOUNT JULIET"
SaleDate,2014-02-05 00:00:00,2014-02-05 00:00:00
SalePrice,65237,65237
LegalReference,20140206-0010606,20140206-0010606
SoldAsVacant,Yes,Yes
OwnerName,NaN,NaN
OwnerAddress,NaN,NaN


## Step 0 - Setup

Verify prerequisites, ensure working copy isolation, check initial shape, and validate SaleDate datetime type.

In [14]:
# Ensure dataframes exist and working copy is isolated
assert 'df_raw' in globals() and 'df_clean' in globals(), "df_raw or df_clean missing"
assert df_clean is not df_raw, "df_clean must be a copy, not the same object as df_raw"

# Ensure change_log is initialized and empty
assert 'change_log' in globals(), "change_log missing"
assert len(change_log) == 0, "change_log should be empty at Step 0"

# Verify shape
print(f"Initial shape: {df_clean.shape}")
assert df_clean.shape == (56477, 19), f"Unexpected shape: {df_clean.shape}"

# Verify SaleDate dtype
is_dt = pd.api.types.is_datetime64_any_dtype(df_clean['SaleDate'])
print(f"SaleDate is datetime: {is_dt}")
assert is_dt, "SaleDate is not datetime type"

print("Step 0 Setup assertions passed successfully.")

Initial shape: (56477, 19)
SaleDate is datetime: True
Step 0 Setup assertions passed successfully.
